# Decorators in Python

My learning notes on **decorators**: how they extend a function's behaviour without changing its original code, and how they can act as a *scheduler* or a *validator*.

## Table of Contents
1. [Prerequisite: functions are objects](#1.-Prerequisite:-functions-are-objects)
2. [The problem decorators solve](#2.-The-problem-decorators-solve)
3. [Building a decorator step by step](#3.-Building-a-decorator-step-by-step)
4. [The `@` syntax](#4.-The-@-syntax)
5. [Common mistake: running logic at decoration time](#5.-Common-mistake:-running-logic-at-decoration-time)
6. [Recap: `*args` and `**kwargs`](#6.-Recap:-*args-and-**kwargs)
7. [Decorators for any number of arguments](#7.-Decorators-for-any-number-of-arguments)
8. [Return values and `functools.wraps`](#8.-Return-values-and-functools.wraps)
9. [Use case: decorator as a scheduler](#9.-Use-case:-decorator-as-a-scheduler)
10. [Use case: decorator as a validator](#10.-Use-case:-decorator-as-a-validator)
11. [Summary / cheat sheet](#11.-Summary-/-cheat-sheet)

## What is a decorator?

A **decorator** is a function that takes another function, adds some extra behaviour around it, and returns a new function, **without modifying the original function's code**.

Typical uses:
- **Extending** functionality (logging, timing, printing headers/footers)
- **Scheduling** (only allow a task to run at certain times)
- **Validating** (check inputs before the real function runs)

## 1. Prerequisite: functions are objects

In Python, a function is a normal object. It can be stored in a variable and **passed as an argument** to another function. Decorators are built on exactly this idea.

In [1]:
def test():
    print("test() function is called")

def demo(func):
    print("demo() function is called")
    func()          # calling the function that was passed in

demo(test)          # pass test (no parentheses!) as an argument

demo() function is called
test() function is called


> `test` is the function object itself. `test()` *runs* it. Here we pass `test` so that `demo` can call it later.

## 2. The problem decorators solve

Suppose we have a simple `withdraw` function:

In [2]:
def withdraw(amount):
    print(f"{amount} withdrawn successfully")

withdraw(25000)

25000 withdrawn successfully


Now we want every withdrawal to also ask for a **PIN first** and print a **receipt after**, without touching `withdraw` itself.

A first attempt is a function `ATM_naive` that takes `withdraw` as an argument:

In [3]:
def ATM_naive(func):
    print("pin")
    func()            # <-- func needs an 'amount', but we are not passing one
    print("receipt")

try:
    ATM_naive(withdraw)
except TypeError as e:
    print("TypeError:", e)

pin
TypeError: withdraw() missing 1 required positional argument: 'amount'


It fails because `withdraw` needs an `amount`, but `ATM_naive` calls it with no arguments. We need a way to **receive the arguments later**, when the user actually calls the function. That is what an *inner (wrapper) function* does.

## 3. Building a decorator step by step

A decorator has three parts:

1. An **outer function** that receives the original function (`func`).
2. An **inner function** (the *wrapper*) that adds extra behaviour and calls `func`.
3. The outer function **returns the wrapper** (without calling it).

In [4]:
def ATM(func):                    # 1. outer function: receives the original function
    def insidefunc(amount):       # 2. inner function: receives the arguments later
        print("pin")
        func(amount)              #    call the original function
        print("receipt")
    return insidefunc             # 3. return the wrapper (no parentheses)

Manual decoration: call the decorator ourselves and keep the returned function.

In [5]:
f1 = ATM(withdraw)    # f1 is now the wrapper (insidefunc)
f1(25000)

pin
25000 withdrawn successfully
receipt


## 4. The `@` syntax

Writing `f1 = ATM(withdraw)` every time is clumsy. Python provides the `@` shortcut:

```python
@ATM
def withdraw(amount): ...
```
is exactly the same as
```python
def withdraw(amount): ...
withdraw = ATM(withdraw)
```

In [6]:
def ATM(func):
    def insidefunc(amount):
        print("pin")
        func(amount)
        print("receipt")
    return insidefunc

@ATM
def withdraw(amount):
    print(f"{amount} withdrawn successfully")

withdraw(25000)

pin
25000 withdrawn successfully
receipt


The same decorator works on a function that takes **no** arguments, as long as the wrapper matches the signature:

In [7]:
def ATM_no_args(func):
    def insidefunc():
        print("pin")
        func()
        print("receipt")
    return insidefunc

@ATM_no_args
def withdraw_cash():
    print("withdrawn successfully")

withdraw_cash()

pin
withdrawn successfully
receipt


## 5. Common mistake: running logic at decoration time

In one of my experiments, I wrote the decorator **without an inner function**:

In [8]:
def ATM_wrong(func):
    print("pin")        # runs immediately, when the function is DEFINED
    func()
    print("receipt")
    return func

@ATM_wrong
def withdraw_wrong():
    print("Money withdrawn")

print("--- function definition is finished, now calling it ---")
withdraw_wrong()

pin
Money withdrawn
receipt
--- function definition is finished, now calling it ---
Money withdrawn


**What went wrong?**
- The `pin` / `withdraw` / `receipt` lines ran **once, at definition time** (the moment `@ATM_wrong` was applied), not each time we call the function.
- When we call `withdraw_wrong()` later, it only prints "Money withdrawn", because the decorator returned the original `func` unchanged.

**Rule:** a decorator must **return a wrapper function** that contains the extra behaviour. The code that should run on every call belongs *inside* the wrapper.

## 6. Recap: `*args` and `**kwargs`

To write a decorator that works for **any** function, we need functions that accept any number of arguments.

- `*args`: collects extra **positional** arguments into a **tuple**
- `**kwargs`: collects extra **keyword** arguments into a **dict**

In [9]:
def add(*args):
    total = 0
    for i in args:
        total += i
    return total

print(add(12, 23, 34, 45, 456, 7))
print(add(12, 23, 34, 45))

577
114


In [10]:
def show_details(**kwargs):
    for key, value in kwargs.items():
        print(key, value)

show_details(Subject="Python", Marks=78)

Subject Python
Marks 78


In [11]:
def demo(*args, **kwargs):
    for i in args:
        print(i)
    for key, value in kwargs.items():
        print(key, value)

demo(12, 23, 24, name="ABC")

12
23
24
name ABC


## 7. Decorators for any number of arguments

Combine the two ideas: the wrapper accepts `*args, **kwargs` and **forwards** them to the original function. This makes the decorator reusable on any function, whatever its parameters.

In [12]:
def intro(func):
    def wrapper(*args, **kwargs):
        print("hi")
        func(*args, **kwargs)          # forward everything unchanged
        print("Welcome to Agentic AI")
    return wrapper

@intro
def sample(*args, **kwargs):
    for data in args:
        print(data)
    for name, value in kwargs.items():
        print(name, value)

sample(20, 30, name="rajesh", education="Btech", position="AI Eng.")

hi
20
30
name rajesh
education Btech
position AI Eng.
Welcome to Agentic AI


## 8. Return values and `functools.wraps`

Two details make a decorator production-quality.

**1. Return the result of the original function.** Without `return`, the wrapper silently swallows the value.

**2. Use `functools.wraps`.** After decoration, the function's name and docstring are replaced by the wrapper's. `@wraps(func)` keeps the original metadata.

In [13]:
from functools import wraps

def log_call(func):
    @wraps(func)                       # keep original name and docstring
    def wrapper(*args, **kwargs):
        print(f"Calling {func.__name__} with {args} {kwargs}")
        result = func(*args, **kwargs)
        return result                  # pass the result back to the caller
    return wrapper

@log_call
def multiply(a, b):
    """Return a * b."""
    return a * b

print(multiply(6, 7))
print("Name :", multiply.__name__)    # 'multiply' (would be 'wrapper' without @wraps)
print("Doc  :", multiply.__doc__)

Calling multiply with (6, 7) {}
42
Name : multiply
Doc  : Return a * b.


## 9. Use case: decorator as a scheduler

A decorator can decide **whether the function is allowed to run** at all. Here the task runs only between **10:00 and 18:59**; otherwise a message is printed.

> The output of the next cells depends on the **current time** when you run them.

In [14]:
import datetime

def schedule_task(func):
    def wrapper():
        hour = datetime.datetime.now().hour
        if 10 <= hour <= 18:
            func()
        else:
            print("Task is not allowed at this time!!!")
    return wrapper

@schedule_task
def task():
    print("Task is running")

task()

Task is running


### Bonus: configurable time window (decorator with arguments)

To choose the allowed hours ourselves, we add **one more outer layer**. `schedule(start, end)` returns the real decorator.

```
schedule(start, end)  ->  decorator(func)  ->  wrapper(*args, **kwargs)
```

In [15]:
def schedule(start, end):
    def decorator(func):
        def wrapper(*args, **kwargs):
            hour = datetime.datetime.now().hour
            if start <= hour <= end:
                return func(*args, **kwargs)
            print(f"Not allowed now (allowed hours: {start}-{end}, current hour: {hour})")
        return wrapper
    return decorator

now = datetime.datetime.now().hour

@schedule(start=now, end=now)          # window includes the current hour -> runs
def allowed_task():
    print("allowed_task is running")

@schedule(start=now + 1, end=now + 2)  # window excludes the current hour -> blocked
def blocked_task():
    print("blocked_task is running")

allowed_task()
blocked_task()

allowed_task is running
Not allowed now (allowed hours: 11-12, current hour: 10)


## 10. Use case: decorator as a validator

A validator decorator **checks the inputs first** and only calls the original function if they are valid.

### Problem 1: voting-age validator
Write a decorator `validator()` that checks a user's age and prints **"Eligible to vote"** if the age is 18 or more.

In [16]:
def validator(func):
    def wrapper(age):
        if isinstance(age, int) and age >= 18:
            func(age)
        else:
            print(f"Age {age}: Not eligible to vote")
    return wrapper

@validator
def vote(age):
    print(f"Age {age}: Eligible to vote")

vote(25)
vote(17)
vote("abc")     # invalid type is rejected too

Age 25: Eligible to vote
Age 17: Not eligible to vote
Age abc: Not eligible to vote


### Problem 2: student-details validator
Allow `Get_Details(Name, RollNo)` to run only if `Name` is a string of **at most 8 characters**.

> My original version used `input()`. Here I use fixed test values so the notebook runs without prompts. To make it interactive, replace them with
> `Name = input("Enter Name: ")` and `RollNo = int(input("Enter Roll: "))`.

In [17]:
def valid_student(func):
    def wrapper(name, roll_no):
        if isinstance(name, str) and len(name) <= 8:
            func(name, roll_no)
        else:
            print("Invalid Name")
    return wrapper

@valid_student
def get_details(name, roll_no):
    print("Student Name is", name)
    print("Student Roll is", roll_no)

get_details("Shubham", 101)         # valid: 7 characters
get_details("Shubham Jadhav", 102)  # invalid: too long
get_details(12345, 103)             # invalid: not a string

Student Name is Shubham
Student Roll is 101
Invalid Name
Invalid Name


## 11. Summary / cheat sheet

- A **decorator** takes a function and returns a **new function** with extra behaviour, leaving the original code untouched.
- Structure: **outer function** (receives `func`) -> **inner `wrapper`** (adds behaviour, calls `func`) -> `return wrapper`.
- `@decorator` above a function is shorthand for `func = decorator(func)`.
- Use `def wrapper(*args, **kwargs)` and `func(*args, **kwargs)` so the decorator works with **any** function.
- **Return** the original function's result from the wrapper, and add `@functools.wraps(func)` to keep its name and docstring.
- Extra behaviour that must run on **every call** belongs **inside** the wrapper, not in the outer function.
- Decorators **with arguments** need one more layer: `schedule(args)` -> `decorator(func)` -> `wrapper(...)`.
- Common uses: **scheduling**, **validation**, logging, timing, authentication, caching.

### Next topics to explore
- Stacking multiple decorators on one function
- Class-based decorators
- Built-in decorators: `@staticmethod`, `@classmethod`, `@property`
- `functools.lru_cache` for caching
- A timing decorator to measure execution speed